# Speech-to-Text ZeroBus Target Table DDL

Creates and maintains the `transcript_events_raw` bronze table in Unity Catalog
and grants access to the lakeLoom service principal.

This notebook is invoked by the `platform_bootstrap` job as the second task.
Parameters are passed from the job definition, which derives catalog/schema from
`${resources.schemas.lakeloom_schema.*}`.

The SPN `application_id` is received from the upstream `ensure_service_principal`
task via task values.

> **Note:** This table uses **liquid clustering** (`CLUSTER BY AUTO`) for
> optimal read performance. **Predictive optimization should remain enabled**
> on the target schema so clustering is applied asynchronously after ZeroBus writes.

In [0]:
%sql
DECLARE OR REPLACE VARIABLE catalog_use STRING;
DECLARE OR REPLACE VARIABLE schema_use STRING;

SET VARIABLE catalog_use = :catalog_use;
SET VARIABLE schema_use = :schema_use;

USE IDENTIFIER(catalog_use || '.' || schema_use);
SELECT current_catalog() AS active_catalog, current_schema() AS active_schema;

In [0]:
%sql
-- Tag all subsequent statements for cost tracking and audit trail.
SET QUERY_TAGS['project'] = 'lakeLoom';
SET QUERY_TAGS['component'] = 'platform_bootstrap';
SET QUERY_TAGS['pipeline'] = 'lakeloom_infra';
EXECUTE IMMEDIATE "SET QUERY_TAGS['catalog'] = '" || catalog_use || "';";
EXECUTE IMMEDIATE "SET QUERY_TAGS['schema'] = '" || schema_use || "';";

In [0]:
%sql
CREATE TABLE IF NOT EXISTS transcript_events_raw
(
  event_id STRING NOT NULL COMMENT 'Producer-generated idempotency key or event identifier',
  session_id STRING COMMENT 'Application session identifier associated with the transcript/event payload',
  project_id STRING COMMENT 'Application project identifier associated with the transcript/event payload',
  user_id STRING COMMENT 'End-user identifier carried in the event payload for attribution beyond the shared SPN actor',
  device_id STRING COMMENT 'Paired-device identifier or label when available',
  event_type STRING NOT NULL COMMENT 'Logical event type such as partial_transcript, final_transcript, audio_uploaded, or client_status',
  event_time TIMESTAMP COMMENT 'Event timestamp supplied by the producer when available',
  ingested_at TIMESTAMP NOT NULL COMMENT 'Server-side ingest timestamp written by the producer or connector',
  transcript_text STRING COMMENT 'Transcript text for transcript-bearing events',
  transcript_language STRING COMMENT 'Language code associated with the transcript text when available',
  source_platform STRING COMMENT 'Origin platform for the event, such as ios',
  workspace_id STRING COMMENT 'Workspace identifier embedded in the event payload when available',
  headers VARIANT COMMENT 'Captured request metadata and non-sensitive headers as semi-structured JSON',
  body VARIANT COMMENT 'Full raw ZeroBus event payload stored as VARIANT for flexible bronze retention',
  CONSTRAINT transcript_events_raw_pk PRIMARY KEY (event_id)
)
USING DELTA
CLUSTER BY AUTO
COMMENT 'Bronze ZeroBus target for lakeLoom transcript and session event ingestion from paired iOS devices'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.enableDeletionVectors' = 'true',
  'delta.enableRowTracking' = 'true',
  'delta.enableVariantShredding' = 'true',
  'quality' = 'bronze',
  'pipeline' = 'lakeloom_platform_bootstrap',
  'description' = 'ZeroBus streaming target table for transcript and pairing-related session events'
);

In [0]:
%sql
SHOW CREATE TABLE transcript_events_raw;

In [0]:
%sql
-- Receive spn_application_id from upstream ensure_service_principal task.
DECLARE OR REPLACE VARIABLE spn_application_id STRING;
SET VARIABLE spn_application_id = :spn_application_id;

SELECT spn_application_id;

In [0]:
%sql
DECLARE OR REPLACE VARIABLE use_catalog_grnt_stmnt STRING DEFAULT
  'GRANT USE CATALOG ON CATALOG ' || catalog_use || ' TO `' || spn_application_id || '`;';

SELECT use_catalog_grnt_stmnt;

In [0]:
%sql
EXECUTE IMMEDIATE use_catalog_grnt_stmnt;

In [0]:
%sql
DECLARE OR REPLACE VARIABLE use_schema_grnt_stmnt STRING DEFAULT
  'GRANT USE SCHEMA ON SCHEMA ' || catalog_use || '.' || schema_use || ' TO `' || spn_application_id || '`;';

SELECT use_schema_grnt_stmnt;

In [0]:
%sql
EXECUTE IMMEDIATE use_schema_grnt_stmnt;

In [0]:
%sql
DECLARE OR REPLACE VARIABLE tbl_grnt_stmnt STRING DEFAULT
  'GRANT MODIFY, SELECT ON TABLE ' || catalog_use || '.' || schema_use || '.transcript_events_raw TO `' || spn_application_id || '`;';

SELECT tbl_grnt_stmnt;

In [0]:
%sql
EXECUTE IMMEDIATE tbl_grnt_stmnt;

In [0]:
%sql
EXECUTE IMMEDIATE 'SHOW GRANTS ON TABLE ' || catalog_use || '.' || schema_use || '.transcript_events_raw';

In [0]:
%sql
SELECT
  'transcript_events_raw_ready' AS check_name,
  'created_or_confirmed' AS status,
  catalog_use || '.' || schema_use || '.transcript_events_raw' AS target_table_name,
  current_timestamp() AS completed_at;